# Week 4 Assignment: Machine Learning for Forecasting
**DSE6230: Forecasting Methods and Applications**

This assignment focuses on using Machine Learning (XGBoost) and Facebook Prophet for forecasting. You will use `mlforecast` and `statsforecast` from Nixtla.

1. **Feature Engineering**: Lags, Rolling Windows, Date Features.
2. **XGBoost**: Training a Gradient Boosted Tree.
3. **Prophet**: Training an Additive Model.
4. **Evaluation**: Compare ML vs Prophet.

In [2]:
!pip install pandas mlforecast statsforecast xgboost datasetsforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.7/261.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.8/501.8 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 3.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from mlforecast.lag_transforms import RollingMean
from statsforecast import StatsForecast
from statsforecast.adapters.prophet import AutoARIMAProphet
from xgboost import XGBRegressor
from datasetsforecast.m4 import M4

## Part 1: Self-Guided Exercises

### Exercise 1.1: Feature Engineering
**Instruction**: Initialize an `MLForecast` object with:
- `models`: `[XGBRegressor()]`
- `freq`: 'D'
- `lags`: [1, 7, 14]
- `lag_transforms`: {1: [window_ops.mean]} (Use Rolling Mean of window 7) *Note: Import `window_ops` first if needed, or skip transforms for basic check.*

In [4]:
# Your Code Here
fcst = MLForecast(
    models=[XGBRegressor()],
    freq='D',
    lags=[1, 7, 14],
    lag_transforms={1: [RollingMean(window_size=7)]}
)

### Exercise 1.2: Prophet
**Instruction**: Use `StatsForecast` to fit a `Prophet` model to a sample dataset (create a dummy df if needed).

In [5]:

from statsforecast.models import AutoARIMA, AutoETS
import numpy as np


# Create dummy dataset
dates = pd.to_datetime(pd.date_range(start='2023-01-01', periods=100, freq='D'))
data = {'unique_id': 'A', 'ds': dates, 'y': pd.Series(range(100)) + np.random.randn(100) * 5}
dummy_df = pd.DataFrame(data)
display(dummy_df.head())

# Fit the StatsForecast model
sf = StatsForecast(models=[AutoARIMA(), AutoETS(season_length=7)], freq='D')
sf.fit(dummy_df)   # keeps unique_id / ds / y

preds = sf.predict(h=30)
preds.head()

,unique_id,ds,y
0,A,2023-01-01,2.489795
1,A,2023-01-02,3.841632
2,A,2023-01-03,11.392624
3,A,2023-01-04,8.517454
4,A,2023-01-05,13.145429


,unique_id,ds,AutoARIMA,AutoETS
0,A,2023-04-11,99.600586,99.583419
1,A,2023-04-12,100.583118,100.565737
2,A,2023-04-13,101.565651,101.548054
3,A,2023-04-14,102.548183,102.530372
4,A,2023-04-15,103.530716,103.512689


## Part 2: Case Study
Download the **M4 Hourly** dataset (or a subset).

1. Train an XGBoost model using `mlforecast`.
2. Train a Prophet model using `statsforecast`.
3. Compare their MAE on a 48-hour horizon.
4. **Analysis**: Which model was faster? Which was more accurate? Why?

In [20]:
# Download the M4 Hourly dataset
group_name = 'Hourly'
Y_df, *_ = M4.load(directory='data', group=group_name)
# Your Code Here

# Subset — all 414 series would make Prophet take a very long time
n_series = 20
ids = Y_df['unique_id'].unique()[:n_series]
df = Y_df[Y_df['unique_id'].isin(ids)].copy()

# Integer ds -> hourly timestamps (Prophet requires datetimes)
df['ds'] = pd.Timestamp('2020-01-01') + pd.to_timedelta(df['ds'] - 1, unit='h')

# Hold out the last 48 hours per series
h = 48
test = df.groupby('unique_id').tail(h)
train = df.drop(test.index)

print(f"{n_series} series | train {len(train)} rows | test {len(test)} rows")

20 series | train 14000 rows | test 960 rows


In [21]:
# XGBoost with mlforecast
import time
fcst = MLForecast(
    models={'XGBoost': XGBRegressor(n_estimators=200, max_depth=6,
                                    learning_rate=0.1, n_jobs=-1)},
    freq='h',
    lags=[1, 2, 3, 24, 48, 168],          # hourly, daily, weekly structure
    lag_transforms={
        1:  [RollingMean(window_size=24)],
        24: [RollingMean(window_size=24)],
    },
    date_features=['hour', 'dayofweek'],
)

t0 = time.perf_counter()
fcst.fit(train)
xgb_preds = fcst.predict(h)
xgb_time = time.perf_counter() - t0

print(f"XGBoost: {xgb_time:.1f}s")

XGBoost: 1.0s


In [22]:
# direct use Prophet
from prophet import Prophet

t0 = time.perf_counter()
rows = []
for uid, g in train.groupby('unique_id'):
    m = Prophet(daily_seasonality=True,    # 24h cycle
                weekly_seasonality=True,   # 168h cycle
                yearly_seasonality=False)  # M4 Hourly series are too short
    m.fit(g[['ds', 'y']])
    future = m.make_future_dataframe(periods=h, freq='h')
    fc = m.predict(future).tail(h)
    rows.append(pd.DataFrame({'unique_id': uid,
                              'ds': fc['ds'].values,
                              'Prophet': fc['yhat'].values}))

prophet_preds = pd.concat(rows, ignore_index=True)
prophet_time = time.perf_counter() - t0

print(f"Prophet: {prophet_time:.1f}s")

Prophet: 9.3s


In [25]:
# AutoARIMA in salesforecast
sf = StatsForecast(
    models=[AutoARIMA(
        season_length=24,
        approximation=True,   # skip exact MLE during search — big speedup
    )],
    freq='h',
    n_jobs=1,
)

t0 = time.perf_counter()
sf.fit(train)
arima_preds = sf.predict(h=h)
arima_time = time.perf_counter() - t0

print(f"AutoARIMA: {arima_time:.1f}s")

/usr/local/lib/python3.13/dist-packages/statsforecast/arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsforecast/arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsforecast/arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsforecast/arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsforecast/arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsforecast/arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsforecast/arima.py:672: UserWarning: possible convergenc

AutoARIMA: 103.2s


In [26]:
res = (test.merge(xgb_preds,     on=['unique_id', 'ds'])
           .merge(arima_preds,   on=['unique_id', 'ds'])
           .merge(prophet_preds, on=['unique_id', 'ds']))

models = ['XGBoost', 'AutoARIMA', 'Prophet']
times  = {'XGBoost': xgb_time, 'AutoARIMA': arima_time, 'Prophet': prophet_time}

summary = pd.DataFrame({
    'Model':    models,
    'Library':  ['mlforecast', 'statsforecast', 'prophet (standalone)'],
    'MAE':      [(res['y'] - res[m]).abs().mean() for m in models],
    'Time (s)': [times[m] for m in models],
}).sort_values('MAE').reset_index(drop=True)
display(summary)

# Per-series MAE — pooled MAE is dominated by high-scale series
per_series = (res.assign(**{f'{m}_ae': (res['y'] - res[m]).abs() for m in models})
                 .groupby('unique_id')[[f'{m}_ae' for m in models]].mean())
display(per_series.round(2))

wins = per_series.idxmin(axis=1).str.replace('_ae', '', regex=False)
print("\nSeries won by each model:")
print(wins.value_counts())

,Model,Library,MAE,Time (s)
0,XGBoost,mlforecast,729.014374,0.971102
1,AutoARIMA,statsforecast,773.958362,103.217442
2,Prophet,prophet (standalone),1239.350337,9.327868


,XGBoost_ae,AutoARIMA_ae,Prophet_ae
unique_id,,,
H1,29.23,28.24,26.20
H10,34.90,13.82,13.93
H100,129.39,62.44,177.00
H101,97.69,110.53,102.57
H102,195.31,121.08,308.09
H103,4676.49,7985.97,11679.43
H104,161.08,122.25,126.94
H105,97.56,153.22,232.92
H106,125.33,157.66,120.73



Series won by each model:
AutoARIMA    10
Prophet       5
XGBoost       5
Name: count, dtype: int64


**In Summary**

**Speed:** XGBoost was 106× faster than AutoARIMA and 10× faster than Prophet. This is architectural， XGBoost trains one global model across all series, so cost stays roughly flat as series are added, while AutoARIMA and Prophet fit one model per series and scale linearly. Notably, AutoARIMA was 11× slower than Prophet here, the opposite of statsforecast's marketing claim, because seasonal ARIMA's stepwise search at season_length=24 is a worst case.

**Accuracy:** The pooled MAE ranking is an artifact. Series H103 alone accounts for 32% of XGBoost's total error and 52% of AutoARIMA's; removing it reverses the ranking, with AutoARIMA clearly ahead. The per-series win count agrees， AutoARIMA took 10 of 20. Pooled MAE on M4 is scale-dominated, which is why the competition uses sMAPE and MASE instead.

**Conclusion:** XGBoost is the practical choice at scale which comparable accuracy for ~1% of the compute. But the correct claim is comparable, not better. AutoARIMA was more accurate per series and simply doesn't scale. Prophet lost on both axes.

**Caveats:** 20 series with one dominating the aggregate makes this ranking unstable. Confirming it would need more series, a scale-free metric (MASE), and multiple random subsets. Global models also tend to improve with more series, so XGBoost's accuracy may rise at full scale while the local models stay flat and untested here.


## Grading Rubric
- **Correctness (40%)**: Does the code run?
- **Code Quality (30%)**: Is the feature engineering step clear?
- **Analysis (30%)**: Comparison of XGBoost vs Prophet.